In [ ]:
%md
**Função Objetivo usada no BHV Consórcio**


%md
Mais completa e personalizada, avalia três métricas (KS, Brier Score, Precision Score).

Usa directions=['maximize', 'minimize', 'maximize'], ou seja, otimiza múltiplos objetivos ao mesmo tempo.

Ideal se você quer controlar bem o trade-off entre acurácia, calibragem e performance prática do modelo.
def objective(trial):
# Hiperparâmetros a serem otimizados
    params = {
    'boosting_type': 'dart',
    'n_estimators': trial.suggest_int('n_estimators', 100, 300),
    'learning_rate': trial.suggest_float('learning_rate', 1e-3, 1e-1, log=True),
    'num_leaves': trial.suggest_int('num_leaves', 5, 50),
    'max_depth': trial.suggest_int('max_depth', 3, 12),
    'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
    "feature_fraction_bynode": trial.suggest_float("feature_fraction_bynode", 0.5, 1.0),
    'min_child_weight': trial.suggest_float('min_child_weight', 1e-3, 1e3, log=True),
    'max_delta_step': trial.suggest_float('max_delta_step', 0.0, 10.0),
    'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 100),
    'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
    'subsample': trial.suggest_float('subsample', 0.5, 1.0),
    'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
    'reg_alpha': trial.suggest_float('reg_alpha', 1e-9, 10.0, log=True),
    'reg_lambda': trial.suggest_float('reg_lambda', 1e-9, 10.0, log=True),
    'max_bin': trial.suggest_int('max_bin', 15, 100),
    }
    # Criar o objeto StratifiedKFold
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores_1 = []
    scores_2 = []
    scores_3 = []
    # Loop para cada fold
    for fold, (train_index, test_index) in enumerate(skf.split(X_train_resampled, y_train_resampled)):
        # Obter os dados de treino e teste para o fold atual
        X_train_cv, X_test_cv = X_train_resampled.iloc[train_index], X_train_resampled.iloc[test_index]
        y_train_cv, y_test_cv = y_train_resampled.iloc[train_index], y_train_resampled.iloc[test_index]

        y_train_cv = y_train_cv.values.ravel().astype(int)
        y_test_cv = y_test_cv.values.ravel().astype(int)

        # Criar e treinar o modelo
        model = lgb.LGBMClassifier(**params, random_state=42, verbosity=-1)
        model.fit(X_train_cv, y_train_cv)
        # Fazer previsões no conjunto de teste
        y_pred = model.predict_proba(X_test_cv)[:, 1]

        # Avaliar o modelo (exemplo: usando AUC)
        scores_1.append(ks2_total(y_test_cv, y_pred)) # roc_auc_score
        scores_2.append(brier_score_loss(y_test_cv, y_pred))
        scores_3.append(average_precision_score(y_test_cv, y_pred))

    scores_1 = np.mean(np.array(scores_1))
    scores_2 = np.mean(np.array(scores_2))
    scores_3 = np.mean(np.array(scores_3))
    # Retornar a média das AUCs como métrica de avaliação
    return scores_1, scores_2, scores_3